# RAG для фінансових звітів

Мета: побудувати RAG-систему, яка відповідає на питання про фінансові показники українських компаній з опорою лише на наданий контекст.

## 1. Імпорти, типізований конфіг та RAG-сервіс

In [9]:
# %pip -q install langchain langchain-openai langchain-community langchain-text-splitters chromadb python-dotenv pydantic

In [10]:
from __future__ import annotations

import os
import re
from collections.abc import Sequence
from enum import Enum
from hashlib import sha256
from typing import Any, Protocol
from uuid import UUID, uuid4

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


class RagSettings(BaseModel):
    """Immutable configuration for the financial reports RAG pipeline."""

    collection_name: str = Field(default="company_financial_statements", min_length=1)
    persist_directory: str = Field(default=".chroma/financial_statements", min_length=1)
    openrouter_model_name: str = Field(default="openai/gpt-4o-mini", min_length=1)
    embedding_model: str = Field(default="openai/text-embedding-3-small", min_length=1)
    openrouter_base_url: str = Field(
        default="https://openrouter.ai/api/v1", min_length=1
    )
    retriever_k: int = Field(default=2, ge=1)
    expected_company_chunks: int = Field(default=4, ge=1)
    chunk_separators: tuple[str, ...] = ("\n\n",)
    chunk_size: int = Field(default=300, ge=1)
    chunk_overlap: int = Field(default=0, ge=0)
    no_info_response: str = Field(
        default=(
            "На основі наданого контексту, інформація про запитувану "
            "компанію або показник відсутня."
        ),
        min_length=1,
    )

    model_config = ConfigDict(frozen=True)


class RuntimeSecrets(BaseModel):
    """Runtime secrets loaded from .env."""

    openrouter_api_key: str = Field(min_length=1)


class Question(BaseModel):
    id: int = Field(ge=1)
    text: str = Field(min_length=1)
    session_id: UUID = Field(default_factory=uuid4)


class Answer(BaseModel):
    request_id: UUID = Field(default_factory=uuid4)
    session_id: UUID
    question_id: int
    question: str
    answer: str


class TestQuestion(Enum):
    TECHNOVA_REVENUE = (1, "Яка виручка у компанії ТехНова Інк за 2024 рік?")
    GREENENERGY_EMPLOYEES = (2, "Скільки працівників у ГрінЕнерджі Солюшнс?")
    RETAILMAX_ROE = (3, "Який показник ROE у РітейлМакс Груп?")
    TECHNOVA_HEADQUARTERS = (4, "Де знаходиться штаб-квартира ТехНова Інк?")
    FINTECH_EBITDA_MARGIN = (5, "Яка маржа EBITDA у ФінТех Дайнамікс?")
    MISSING_COMPANY_NET_PROFIT = (6, "Який чистий прибуток у ЕкоПакування Солюшнс?")

    @property
    def question_id(self) -> int:
        return self.value[0]

    @property
    def text(self) -> str:
        return self.value[1]

    def to_question(self, session_id: UUID | None = None) -> Question:
        if session_id is None:
            return Question(id=self.question_id, text=self.text)
        return Question(id=self.question_id, text=self.text, session_id=session_id)


class RetrieverProtocol(Protocol):
    def __or__(self, other: Any) -> Any: ...

    def invoke(
        self, input: str, config: Any | None = None, **kwargs: Any
    ) -> list[Document]: ...


class AnswerChainProtocol(Protocol):
    def invoke(self, input: str, config: Any | None = None, **kwargs: Any) -> str: ...


def load_runtime_secrets() -> RuntimeSecrets:
    load_dotenv()
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY не знайдено у .env")
    return RuntimeSecrets(openrouter_api_key=api_key)


def extract_company(text: str) -> str:
    match = re.search(r"^(.+?) за 2024 рік:", text.strip())
    return match.group(1).strip() if match else "Невідомо"


def format_documents(documents: Sequence[Document]) -> str:
    return "\n\n".join(document.page_content for document in documents)


def make_document_id(document: Document) -> str:
    source = document.metadata.get("source", "unknown_source")
    company = extract_company(document.page_content)
    raw_id = f"{source}:{company}:{document.page_content}"
    return sha256(raw_id.encode("utf-8")).hexdigest()


class FinancialReportRAG:
    """Builds and serves answers from the financial reports vector store."""

    def __init__(self, settings: RagSettings, api_key: str) -> None:
        self.settings = settings
        self.api_key = api_key
        self._documents: list[Document] | None = None
        self._vectorstore: Chroma | None = None
        self._retriever: RetrieverProtocol | None = None
        self._chain: AnswerChainProtocol | None = None

    @property
    def documents(self) -> list[Document]:
        if self._documents is None:
            raise RuntimeError("Спочатку підготуйте документи через prepare()")
        return self._documents

    @property
    def vectorstore(self) -> Chroma:
        if self._vectorstore is None:
            raise RuntimeError(
                "Спочатку створіть vectorstore через build() або prepare()"
            )
        return self._vectorstore

    @property
    def retriever(self) -> RetrieverProtocol:
        if self._retriever is None:
            raise RuntimeError(
                "Спочатку створіть retriever через build() або prepare()"
            )
        return self._retriever

    @property
    def chain(self) -> AnswerChainProtocol:
        if self._chain is None:
            raise RuntimeError(
                "Спочатку створіть RAG chain через build() або prepare()"
            )
        return self._chain

    def split_documents(self, document: Document) -> list[Document]:
        splitter = RecursiveCharacterTextSplitter(
            separators=list(self.settings.chunk_separators),
            chunk_size=self.settings.chunk_size,
            chunk_overlap=self.settings.chunk_overlap,
        )
        documents = splitter.split_documents([document])

        expected = self.settings.expected_company_chunks
        if len(documents) != expected:
            raise ValueError(f"Очікувалось {expected} чанки, отримано {len(documents)}")

        return documents

    def _tag_company_metadata(self, documents: Sequence[Document]) -> list[Document]:
        return [
            Document(
                page_content=document.page_content,
                metadata={
                    **document.metadata,
                    "document_id": make_document_id(document),
                    "company": extract_company(document.page_content),
                },
            )
            for document in documents
        ]

    def create_embeddings(self) -> OpenAIEmbeddings:
        return OpenAIEmbeddings(
            model=self.settings.embedding_model,
            api_key=self.api_key,
            base_url=self.settings.openrouter_base_url,
        )

    def create_vectorstore(self, documents: Sequence[Document]) -> Chroma:
        document_list = list(documents)
        self._vectorstore = Chroma.from_documents(
            documents=document_list,
            embedding=self.create_embeddings(),
            ids=[document.metadata["document_id"] for document in document_list],
            collection_name=self.settings.collection_name,
            persist_directory=self.settings.persist_directory,
        )
        return self._vectorstore

    def create_retriever(self, vectorstore: Chroma | None = None) -> RetrieverProtocol:
        active_vectorstore = vectorstore or self._vectorstore
        if active_vectorstore is None:
            raise RuntimeError("Спочатку створіть vectorstore")

        self._retriever = active_vectorstore.as_retriever(
            search_kwargs={"k": self.settings.retriever_k}
        )
        return self._retriever

    def create_prompt(self) -> ChatPromptTemplate:
        return ChatPromptTemplate.from_template(
            f"""Ти аналітик фінансових звітів. Відповідай лише на основі контексту.

Контекст:
{{context}}

Питання: {{question}}

Правила:
- Відповідай українською.
- Наводь конкретні цифри або факти з контексту.
- Якщо назва компанії або потрібний показник не згадується у контексті, не вигадуй дані й конкретно скажи, що інформація про запитувану компанію або показник відсутня. Базове формулювання: "{self.settings.no_info_response}"

Відповідь:
"""
        )

    def create_llm(self) -> ChatOpenAI:
        return ChatOpenAI(
            model=self.settings.openrouter_model_name,
            temperature=0,
            api_key=self.api_key,
            base_url=self.settings.openrouter_base_url,
        )

    def create_chain(
        self, retriever: RetrieverProtocol | None = None
    ) -> AnswerChainProtocol:
        active_retriever = retriever or self._retriever
        if active_retriever is None:
            raise RuntimeError("Спочатку створіть retriever")

        self._chain = (
            {
                "context": active_retriever | format_documents,
                "question": RunnablePassthrough(),
            }
            | self.create_prompt()
            | self.create_llm()
            | StrOutputParser()
        )
        return self._chain

    def build(self, documents: Sequence[Document]) -> FinancialReportRAG:
        self._documents = self._tag_company_metadata(documents)
        vectorstore = self.create_vectorstore(self._documents)
        retriever = self.create_retriever(vectorstore)
        self.create_chain(retriever)
        return self

    def prepare(self, document: Document) -> list[Document]:
        self.build(self.split_documents(document))
        return self.documents

    def answer_question(self, question: Question) -> Answer:
        if self._chain is None:
            raise RuntimeError("Спочатку створіть RAG chain")

        response = self._chain.invoke(question.text)
        return Answer(
            session_id=question.session_id,
            question_id=question.id,
            question=question.text,
            answer=response.strip(),
        )


settings = RagSettings()
secrets = load_runtime_secrets()
rag = FinancialReportRAG(settings=settings, api_key=secrets.openrouter_api_key)

## 2. Вхідні фінансові дані

In [11]:
raw_text = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

base_doc = Document(
    page_content=raw_text, metadata={"source": "financial_reports_2024"}
)

## 3. Підготовка RAG pipeline через prepare()

In [12]:
splits = rag.prepare(base_doc)

print("Кількість чанків:", len(splits))
print("Компанії:", [doc.metadata["company"] for doc in splits])
print("Document IDs:", [doc.metadata["document_id"] for doc in splits])
print("Колекція:", settings.collection_name)
print("Persist directory:", settings.persist_directory)
print("Retriever k:", settings.retriever_k)

Кількість чанків: 4
Компанії: ['ТехНова Інк', 'ГрінЕнерджі Солюшнс', 'РітейлМакс Груп', 'ФінТех Дайнамікс']
Document IDs: ['c244a7e916406c43b3b4ee40c9428c34071252d47728fdda7931fcb1bb2bf04f', '1f2741d198379a0dc40d47be91f5996c6f5be02bb06473871ad8149005eff6cb', '4d6fc75ad84fb970ad41e08a2ae1aad68e3165a3fbae31426097870abbb5a143', '24944cd9156eff4bc20256ab30a7177ead0845a05af44a9a9d61ebc741969563']
Колекція: company_financial_statements
Persist directory: .chroma/financial_statements
Retriever k: 2


## 4. Векторне сховище (Chroma)

In [13]:
vectorstore = rag.vectorstore

print("Колекція:", settings.collection_name)
print("Кількість документів у сховищі:", len(splits))

Колекція: company_financial_statements
Кількість документів у сховищі: 4


## 5. Налаштування retriever

In [14]:
retriever = rag.retriever

## 6. RAG ланцюг

In [15]:
rag_chain = rag.chain

## 7. Тестові питання та відповіді

In [16]:
session_id = uuid4()
questions = [
    test_question.to_question(session_id=session_id) for test_question in TestQuestion
]

print(f"Session ID: {session_id}")
print("-" * 70)

for question in questions:
    result = rag.answer_question(question)
    print(f"Request ID: {result.request_id}")
    print(f"Питання {result.question_id}: {result.question}")
    print(f"Відповідь: {result.answer}")
    print("-" * 70)

Session ID: 1bbb7dda-33eb-47c8-8dae-d2fba3a0d86b
----------------------------------------------------------------------
Request ID: 6a6e9b63-3dff-4238-9662-48ea5cb08039
Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка у компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------
Request ID: 79a1be99-f834-4f65-bf61-ceb61b3b74df
Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------
Request ID: 97a2bed4-1561-4d7c-b13d-07d5dbcca1c0
Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: На основі наданого контексту, ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------
Request ID: 1b008913-960c-4f60-bb31-cb1214584f60
Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знахо